In [2]:
import requests

airport_url = "https://en.wikipedia.org/wiki/List_of_international_airports_by_country"

headers = {"User-Agent": "airport-codes-scraper/0.1 (your@email.com)"}
airport_resp = requests.get(airport_url, headers=headers, timeout=30)
airport_resp.raise_for_status()


In [5]:
from io import StringIO
import pandas as pd

tables = pd.read_html(StringIO(airport_resp.text))
print(len(tables))

230


In [11]:
from bs4 import BeautifulSoup
from io import StringIO
import pandas as pd

soup = BeautifulSoup(airport_resp.text, "lxml")

frames = []

for table in soup.select("table.wikitable"):
    df = pd.read_html(StringIO(str(table)))[0]

    df["country"] = table.find_previous(["h3", "h4"]).get_text(strip=True)
    df["continent"] = table.find_previous("h2").get_text(strip=True)

    frames.append(df)

df_airports = pd.concat(frames, ignore_index=True)

In [12]:
df_airports.head(4)

,Location,Airport,IATA Code,country,continent,Passenger Role,Provinces/Territories,State,ICAO Code,State/Union Territory
0,Algiers,Houari Boumediene Airport,ALG,Algeria,Africa,NaN,NaN,NaN,NaN,NaN
1,Annaba,Rabah Bitat Airport,AAE,Algeria,Africa,NaN,NaN,NaN,NaN,NaN
2,Batna,Batna Airport,BLJ,Algeria,Africa,NaN,NaN,NaN,NaN,NaN
3,Béjaïa,Soummam Airport,BJA,Algeria,Africa,NaN,NaN,NaN,NaN,NaN


In [15]:
df_airports_new = df_airports[["Location", "Airport", "IATA Code", "ICAO Code", "country", "continent"]]

In [17]:
df_airports_new.head(10)

,Location,Airport,IATA Code,ICAO Code,country,continent
0,Algiers,Houari Boumediene Airport,ALG,NaN,Algeria,Africa
1,Annaba,Rabah Bitat Airport,AAE,NaN,Algeria,Africa
2,Batna,Batna Airport,BLJ,NaN,Algeria,Africa
3,Béjaïa,Soummam Airport,BJA,NaN,Algeria,Africa
4,Chlef,Chlef International Airport,CFK,NaN,Algeria,Africa
5,Constantine,Mohamed Boudiaf International Airport,CZL,NaN,Algeria,Africa
6,Ghardaïa,Noumérat – Moufdi Zakaria Airport,GHA,NaN,Algeria,Africa
7,Jijel,Jijel Ferhat Abbas Airport,GJL,NaN,Algeria,Africa
8,Oran,Oran Es Senia Airport,ORN,NaN,Algeria,Africa
9,Oumache,Biskra Airport,BSK,NaN,Algeria,Africa


In [23]:
df_airports_new.shape

(1552, 6)

In [21]:
df_airports_new.info() 

<class 'pandas.DataFrame'>
RangeIndex: 1552 entries, 0 to 1551
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Location   1551 non-null   str  
 1   Airport    1551 non-null   str  
 2   IATA Code  1551 non-null   str  
 3   ICAO Code  4 non-null      str  
 4   country    1552 non-null   str  
 5   continent  1552 non-null   str  
dtypes: str(6)
memory usage: 72.9 KB


In [ ]:
# Amount of rows missing IATA Code
df_airports["IATA Code"].isna().sum()

np.int64(1)

In [ ]:
# Amount of rows NOT missing ICAO Code
df_airports_new["ICAO Code"].notna().sum()

np.int64(4)

In [ ]:
# Rows missing IATA Code 
df_airports_new[df_airports_new["IATA Code"].isna() | (df_airports_new["IATA Code"]== "")]

,Location,Airport,IATA Code,ICAO Code,country,continent
942,NaN,NaN,NaN,NaN,Vietnam,Asia


In [ ]:
# Rows NOT missing ICAO Code
df_airports_new[df_airports_new["ICAO Code"].notna()]

,Location,Airport,IATA Code,ICAO Code,country,continent
621,Bishkek,Manas International Airport,FRU,UCFM,Kyrgyzstan,Asia
622,Issyk-Kul,Issyk-Kul International Airport,IKU,UCFL,Kyrgyzstan,Asia
623,Karakol,Karakol International Airport,IKG,UCFP,Kyrgyzstan,Asia
624,Osh,Osh Airport,OSS,UCFO,Kyrgyzstan,Asia


In [ ]:
# Duplicate rows
df.duplicated().sum()

np.int64(0)

In [36]:
df_airports_new[df_airports_new["Location"].isna() | (df_airports_new["Location"]== "")]

,Location,Airport,IATA Code,ICAO Code,country,continent
942,NaN,NaN,NaN,NaN,Vietnam,Asia


In [ ]:
df_airports_new[df_airports_new["Airport"].isna() | (df_airports_new["Airport"] == "")]

,Location,Airport,IATA Code,ICAO Code,country,continent
942,NaN,NaN,NaN,NaN,Vietnam,Asia


## EDA discoveries

- missing value found in columns Location, Airport, IATA Code and ICAO Code
- 4 countries have ICAO Code
- The one missing value in Location, Airport and IATA Code are all in the same row